# Random Forest Model Performance

This notebook reviews the latest random forest outputs produced by `model_training.ipynb`.
It does not retrain models. It loads the timestamped metrics, predictions/residuals, feature importances, and hyperparameters from the external output folder.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from importlib import import_module

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
try:
    from IPython.display import display
except Exception:
    display = print

In [ ]:
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pipeline" / "config").exists()
)
sys.path.append(str(REPO_ROOT))

from pipeline.utils.paths import load_paths, path_value

rf_utils = import_module("pipeline.3_epvi_prediction.utils")
latest_file = rf_utils.latest_file

PATHS = load_paths()
MODEL_OUT_DIR = path_value(PATHS, "external_data_root") / "outputs" / "epvi_prediction" / "random_forest"

metrics_path = latest_file(MODEL_OUT_DIR, "rf_epvi_metrics_*.csv")
predictions_path = latest_file(MODEL_OUT_DIR, "rf_epvi_predictions_residuals_*.csv")
importances_path = latest_file(MODEL_OUT_DIR, "rf_epvi_feature_importances_*.csv")
params_path = latest_file(MODEL_OUT_DIR, "rf_epvi_best_params_*.json")
permutation_path = latest_file(MODEL_OUT_DIR, "rf_epvi_permutation_importances_*.csv")

required = {
    "metrics": metrics_path,
    "predictions/residuals": predictions_path,
    "feature importances": importances_path,
    "best params": params_path,
}
missing = [name for name, path in required.items() if path is None]
if missing:
    raise FileNotFoundError(
        f"Missing model output files in {MODEL_OUT_DIR}: {missing}. Run model_training.ipynb first."
    )

print("model output folder:", MODEL_OUT_DIR)
print("metrics:", metrics_path)
print("predictions:", predictions_path)
print("importances:", importances_path)
print("params:", params_path)
if permutation_path is not None:
    print("permutation importances:", permutation_path)

In [ ]:
metrics = pd.read_csv(metrics_path)
predictions = pd.read_csv(predictions_path)
importances = pd.read_csv(importances_path)
with open(params_path, "r", encoding="utf-8") as f:
    best_params = json.load(f)

if permutation_path is not None:
    permutation_importances = pd.read_csv(permutation_path)
else:
    permutation_importances = pd.DataFrame()

# Stable target order: best performing first by out-of-fold R2.
target_order = metrics.sort_values("oof_r2", ascending=False)["target"].tolist()
metrics["target"] = pd.Categorical(metrics["target"], categories=target_order, ordered=True)
predictions["target"] = pd.Categorical(predictions["target"], categories=target_order, ordered=True)
importances["target"] = pd.Categorical(importances["target"], categories=target_order, ordered=True)

print("targets:", target_order)
display(metrics.sort_values("target"))

## Overall Metrics

Out-of-fold metrics are the main performance estimate. In-sample metrics are included only as a rough overfitting diagnostic.

In [ ]:
metric_cols = ["oof_r2", "oof_mae", "oof_rmse", "oof_spearman", "in_sample_r2"]
plot_df = metrics.sort_values("oof_r2", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_df.plot.barh(x="target", y="oof_r2", ax=axes[0], legend=False, color="#356f8c")
axes[0].set_title("Out-of-fold R2")
axes[0].axvline(0, color="black", linewidth=0.8)

plot_df.plot.barh(x="target", y="oof_rmse", ax=axes[1], legend=False, color="#8c5a35")
axes[1].set_title("Out-of-fold RMSE")

plot_df.plot.barh(x="target", y="oof_spearman", ax=axes[2], legend=False, color="#4f7f45")
axes[2].set_title("Out-of-fold Spearman")
axes[2].axvline(0, color="black", linewidth=0.8)

for ax in axes:
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

display(metrics[["target", "n_rows", "n_predictors", *metric_cols]].sort_values("oof_r2", ascending=False))

## Observed vs. Predicted

The scatter plots use out-of-fold predictions, so each point is predicted by a model that did not train on that parish.

In [ ]:
n_targets = len(target_order)
fig, axes = plt.subplots(n_targets, 2, figsize=(12, 4 * n_targets))
if n_targets == 1:
    axes = np.array([axes])

for row, target in enumerate(target_order):
    df_t = predictions[predictions["target"] == target].copy()

    ax = axes[row, 0]
    ax.scatter(df_t["observed"], df_t["predicted_oof"], s=16, alpha=0.55, color="#356f8c")
    lo = min(df_t["observed"].min(), df_t["predicted_oof"].min())
    hi = max(df_t["observed"].max(), df_t["predicted_oof"].max())
    ax.plot([lo, hi], [lo, hi], color="black", linewidth=0.9)
    ax.set_title(f"{target}: observed vs OOF predicted")
    ax.set_xlabel("Observed")
    ax.set_ylabel("Predicted")
    ax.grid(alpha=0.25)

    ax = axes[row, 1]
    ax.hist(df_t["residual_oof"].dropna(), bins=40, color="#8c5a35", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.9)
    ax.set_title(f"{target}: OOF residuals")
    ax.set_xlabel("Observed - predicted")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## Largest Residuals

These parishes are useful starting points for the residual-explanation stage with detailed administrative indicators.

In [ ]:
residual_tables = []
for target in target_order:
    df_t = predictions[predictions["target"] == target].copy()
    df_t["abs_residual_oof"] = df_t["residual_oof"].abs()
    residual_tables.append(
        df_t.sort_values("abs_residual_oof", ascending=False)
        [["target", "ID_norm", "name", "observed", "predicted_oof", "residual_oof", "abs_residual_oof"]]
        .head(15)
    )

largest_residuals = pd.concat(residual_tables, ignore_index=True)
display(largest_residuals)

## Feature Importances

Impurity importances are fast and useful for a first pass, but they can favor continuous or high-cardinality predictors. If the training notebook was run with permutation importances enabled, those are shown below as a more conservative diagnostic.

In [ ]:
TOP_N = 15
for target in target_order:
    df_t = importances[importances["target"] == target].sort_values("impurity_importance", ascending=True).tail(TOP_N)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(df_t["feature"], df_t["impurity_importance"], color="#356f8c")
    ax.set_title(f"{target}: top impurity importances")
    ax.set_xlabel("Importance")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
if permutation_importances.empty:
    print("No permutation-importance file found. Set RUN_PERMUTATION_IMPORTANCE = True in model_training.ipynb if needed.")
else:
    for target in target_order:
        df_t = (
            permutation_importances[permutation_importances["target"] == target]
            .sort_values("permutation_importance_mean", ascending=True)
            .tail(15)
        )
        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.barh(df_t["feature"], df_t["permutation_importance_mean"], color="#4f7f45")
        ax.set_title(f"{target}: top permutation importances")
        ax.set_xlabel("Mean R2 decrease")
        ax.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()

## Tuned Hyperparameters

In [ ]:
params_df = pd.DataFrame(best_params).T
params_df.index.name = "target"
display(params_df)